# BTC EKF approach

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from threading import Thread
from queue import Queue, Full, Empty
from collections import deque

# --- Order‑book crawler ---
def get_order_book(symbol="BTCUSDC", limit=10):
    url = f"https://api.binance.com/api/v3/depth?symbol={symbol}&limit={limit}"
    data = requests.get(url).json()
    bids = np.array(data['bids'], dtype=float)
    asks = np.array(data['asks'], dtype=float)
    obi = (bids[:,1].sum() - asks[:,1].sum()) / (bids[:,1].sum() + asks[:,1].sum() + 1e-9)
    mid_price = (bids[0,0] + asks[0,0]) / 2
    return mid_price, obi

# --- Parameters ---
symbol          = "BTCUSDC"
limit           = 10
dt              = 1.0                # data fetch period (s)
burn_in_samples = 10
vol_window      = 10
T               = 180

# Influence coefficients
c_obi =  1.0
c_vol = -0.5

# --- Shared queue for the latest data ---
data_q = Queue(maxsize=1)

def data_fetcher():
    """Continuously poll Binance and keep only the latest sample in data_q."""
    while True:
        try:
            price, obi = get_order_book(symbol, limit)
            # drop old if queue full
            try:
                data_q.put((price, obi), block=False)
            except Full:
                data_q.get_nowait()
                data_q.put((price, obi), block=False)
        except Exception as e:
            print("Fetcher error:", e)
        time.sleep(dt)

# start background thread
thread = Thread(target=data_fetcher, daemon=True)
thread.start()

# --- Burn‑in to initialize state ---
price_buf = deque(maxlen=vol_window)
obi_buf   = []
vol_buf   = []

print(f"Burn‑in: waiting for {burn_in_samples} samples…")
for _ in range(burn_in_samples):
    price, obi = data_q.get()  # blocks until fetcher puts data
    price_buf.append(price)
    obi_buf.append(obi)
    arr = np.array(price_buf)
    if arr.size >= 2:
        rets = np.diff(arr) / arr[:-1]
        vol  = np.std(rets)
    else:
        vol = 0.0
    vol_buf.append(vol)

# initial state
t_idx  = np.arange(len(price_buf)) * dt
slope, _ = np.polyfit(t_idx, list(price_buf), 1)
x = np.array([price_buf[-1], slope])
P = np.diag([np.var(price_buf), np.var(obi_buf)+np.var(vol_buf)]) * 10

print("Init x =", x)
print("Init P diag =", np.diag(P))

# --- EKF matrices ---
A = np.array([[1, dt],[0,1]])
B = np.array([[     0,      0],
              [c_obi*dt, c_vol*dt]])
H = np.array([[1,0]])
Q = np.eye(2)*0.1
R = np.array([[25]])

# storage
estimates = []
obs_prices = []
obis       = []
vols       = []

print(f"Running EKF for {T} steps…")
for _ in range(T):
    price, obi = data_q.get()
    price_buf.append(price)
    arr = np.array(price_buf)
    if arr.size >= 2:
        rets = np.diff(arr) / arr[:-1]
        vol  = np.std(rets)
    else:
        vol = 0.0

    # Predict
    u      = np.array([obi, vol])
    x_pred = A @ x + B @ u
    P_pred = A @ P @ A.T + Q

    # Update
    y_pred     = H @ x_pred
    innov      = price - y_pred
    S          = H @ P_pred @ H.T + R
    K          = P_pred @ H.T @ np.linalg.inv(S)
    x          = x_pred + (K.flatten() * innov)
    P          = (np.eye(2) - K @ H) @ P_pred

    # record
    estimates.append(x.copy())
    obs_prices.append(price)
    obis.append(obi)
    vols.append(vol)

# --- Plot ---
df = pd.DataFrame(estimates, columns=['price_est','vel_est'])
df['obs_price'] = obs_prices
df['obi']       = obis
df['vol']       = vols

plt.figure(figsize=(12,4))
plt.plot(df['obs_price'], alpha=0.5, label='Observed')
plt.plot(df['price_est'],      label='EKF Est.')
plt.legend(); plt.grid(); plt.title("Price")

plt.figure(figsize=(12,3))
plt.plot(df['vel_est'], label='Velocity'); plt.axhline(0, ls='--')
plt.legend(); plt.grid(); plt.title("Velocity")

plt.figure(figsize=(12,3))
plt.plot(df['obi'], label='OBI')
plt.plot(df['vol'], label='Vol')
plt.legend(); plt.grid(); plt.title("Inputs")

plt.show()
